## **Multi-Head Attention**

In [1]:
import torch 
import math
import torch.nn as nn
import torch.nn.functional as F

## **Input Embeddings**

In [2]:
##initialize the vocab size and its model dimension

class InputEmbeddings(nn.Module):
    ##initialize 
    def __init__(self, vocab_size: int, d_model:int)->None:
        super().__init__()
        # Set the model dimensionality and vocabulary size
        self.d_model = d_model
        self.vocab_size = vocab_size
        #forming Embedding Matrix with shape = (vocab_size, d_model)
        # Instantiate the embedding layer
        self.embeddings = nn.Embedding(vocab_size, d_model)
    
    ##forword feed
    def forward(self, x):
        # Return the embeddings multiplied by the square root of d_model
        ##scaled up factors to scale embeddings
        scaled_factor = math.sqrt(self.d_model)
        #dense vector representation
        tensor = self.embeddings(x)
        return tensor*scaled_factor

## **Positional Encoding**

In [3]:
##positional encoding layers
##define the positional encoding class by inherting from torch module
class PositionalEncoding(nn.Module):
    ##intialize
    def __init__(self, d_model:int, max_seq_length:int):
        super().__init__()
        # Create a matrix of zeros of dimensions max_seq_length by d_model
        ##initialize the positional embeddings (pe) to zeros
        pe = torch.zeros(max_seq_length, d_model)
        ##create the tensor of positions for each token in the sequence then it transform using unsqueeze
        ##so that it can be used in positional encoding calculations. unsqueeze(1)- converts into tensor.
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        ##division term in sin and cosine function
        div_term = torch.exp(torch.arange(0,d_model, 2, dtype=torch.float)* -(math.log(10000.0)/d_model))

        ##sin and cosine calculation
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        ##store pe without making learnable parameter during training
        self.register_buffer('pe', pe.unsqueeze(0))

    ##adding input token embeddings (X_input_token) with positional embedding (pe)
    def forward(self, x_embed_token):
        ##operation: element-wise addition
        #x_input_token + positional_encoding
        #token_representation + position_representation
        return x_embed_token+self.pe[:, :x_embed_token.size(1)]

## **Multi-Head Attention**

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads): ##num_heads is number of attention head, each handling embeddings of size head_dim
        super().__init__()
        ##applying constraint on head_dim using (assert) method
        assert d_model%num_heads==0, "d_model must be divisible by num heads"
        ##initializing the d_model and num_heads
        # Calculate the dimensions each head will process
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model//num_heads

        ##initialize three embedding matrices of equal dimension
        ##Three linear layers are defined for the attention inputs
        ##bias = False, no impact on performance while reducing complexity (only for inputs): Q,K,V layers eliminates the bias term,
        ##reducing model parameters without impacting the ability to capture relationships.
        # Define the three input layers and one output layer
        self.query_linear = nn.Linear(d_model, d_model, bias=False) 
        self.key_linear = nn.Linear(d_model, d_model, bias=False)
        self.values_linear = nn.Linear(d_model, d_model, bias=False)
        ##final concatenated output layer
        self.output_linear = nn.Linear(d_model, d_model)
    
    ##Three different helper method
    ##split and transform the input embeddings between the attention heads
    def split_heads(self, x, batch_size):
        seq_length = x.size(1)
        # Split the input embeddings and permute
        x=x.reshape(batch_size, seq_length, self.num_heads, self.head_dim)
        ##rearrange the position of vector/element using index
        return x.permute(0,2,1,3)
    
    ##compute attention weights using F.softmax
    # Compute scaled dot-product attention
    def compute_attention(self, query, key, value, mask=None):
        ##matrix multiplication using matmul method of torch
        ##requires to transpose Key-matrix, and calculate the attention weights inside each head using softmax
        scores = torch.matmul(query, key.transpose(-2, -1))/(self.head_dim**0.5)
        if mask is not None:
            scores = scores.masked_fill(mask==0, float('-inf'))
        ##calculating attention weights
        attention_weights = F.softmax(scores, dim=-1)
        ##return these attention weights matrix multiplied by the value matrix
        return torch.matmul(attention_weights, value)

    ##transform the attention weights back into the original embedding shape
    def combine_heads(self,x, batch_size):
        x = x.permute(0,2,1,3).contiguous()
        # Combine heads back to (batch_size, seq_length, d_model)
        return x.view(batch_size, -1, self.d_model)
    
    ##forward method 
    def forward(self, query, key, value, mask =None):
        batch_size = query.size(0)

        Query = self.split_heads(self.query_linear(query), batch_size)
        Key = self.split_heads(self.key_linear(key), batch_size)
        Values = self.split_heads(self.values_linear(value), batch_size)

        ##attention weights
        attention_weights = self.compute_attention(Query, Key, Values, mask)

        ##output
        output = self.combine_heads(attention_weights, batch_size)
        return self.output_linear(output)




## **FeedForward Sub-Layer**

In [5]:
##feed-forward sub-layer
class FeedForwardSubLayer(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        ##define the fully-connected layers and activation
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
    
    def forward(self,x):
        # Pass the input through the layers and activation
        return self.fc2(self.relu(self.fc1(x)))

In [ ]:
# # Instantiate the FeedForwardSubLayer and apply it to x
# feed_forward = FeedForwardSubLayer(512,2048)
# output = feed_forward(x)
# print(f"Input shape: {x.shape}")
# print(f"Output shape: {output.shape}")

## **Encoder Layer**: This layer puts Multi-Head Attention mechanism and FeedForward Sub-layer together.

In [6]:
##encoder layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, num_heads, dropout):
        super().__init__()
        ##instantiate the layer
        ##multi-head attention for self-attention
        self.self_atten = MultiHeadAttention(d_model, num_heads)
        self.ff_sublayer = FeedForwardSubLayer(d_model, d_ff)
        ##Normaliztion layer
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.Dropout = nn.Dropout(dropout)
    
    def forward(self, x, src_mask):
        ##complete the forward method
        atten_output = self.self_atten(x,x,x, src_mask)
        x = self.norm1(x+self.Dropout(atten_output))
        ff_output = self.ff_sublayer(x)
        x = self.norm2(x+self.Dropout(ff_output))
        return x

## **Encoder Transformer Body**

In [7]:
##encoder transformer
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length):
        super().__init__()
        ##instantiate inputembedding layer, positional encoding, encoder layer
        self.embeddings = InputEmbeddings(vocab_size=vocab_size, d_model=d_model)
        self.positional_encoding = PositionalEncoding(d_model=d_model, max_seq_length=max_seq_length)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model=d_model, d_ff=d_ff, num_heads=num_heads, dropout=dropout) for _ in range(num_layers)
        ])
    
    def forward(self, x, src_mask):
        x = self.embeddings(x)
        x = self.positional_encoding(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return x

## **Classfication Head**

In [6]:
#classification head
class ClassificationHead(nn.Module):
    def __init__(self, d_model, num_classes):
        super().__init__()
        self.fc = nn.Linear(d_model, num_classes)
    
    def forward(self, x):
        logit = self.fc(x)
        return F.log_softmax(logit, dim=1)

## **Regression Head**

In [7]:
##regression head
class RegressionHead(nn.Module):
    def __init__(self, d_model, output_dim):
        super().__init__()
        self.fc = nn.Linear(d_model, output_dim)
    
    def forward(self, x):
        return self.fc(x)

In [ ]:
# # Instantiate the encoder transformer's body and head
# encoder = TransformerEncoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, seq_length)
# classifier = ClassifierHead(d_model, num_classes)


# # Complete the forward pass 
# output = encoder(input_sequence, src_mask)
# classification = classifier(output)
# print(f"Classification outputs for a batch of {batch_size} sequences:\n{classification}")
# print(f"Encoder output shape: {output.shape}\nClassification head output shape: {classification.shape}")

## **Decoder Layer**

In [12]:
seq_length= 3
# Create a Boolean matrix to mask future tokens
tgt_mask = (1-torch.triu(torch.ones(1,seq_length, seq_length), diagonal=1)).bool()

print("Masked Matrix\n", tgt_mask)

Masked Matrix
 tensor([[[ True, False, False],
         [ True,  True, False],
         [ True,  True,  True]]])


In [13]:
## Decoder layer class
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff_sublayer = FeedForwardSubLayer(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, tgt_mask):
        # Perform the attention calculation
        attn_output = self.self_attn(x,x,x,tgt_mask)
        # Apply dropout and the first layer normalization
        x = self.norm1(x + self.dropout(attn_output))
        # Pass through the feed-forward sublayer
        ff_output = self.ff_sublayer(x)
        # Apply dropout and the second layer normalization
        x = self.norm2(x + self.dropout(ff_output))
        return x

In [14]:
##Decoder Transformer
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length):
        super(TransformerDecoder, self).__init__()
        self.embedding = InputEmbeddings(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        # Define the list of decoder layers and linear layer
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        # Define a linear layer to project hidden states to likelihoods
        self.fc = nn.Linear(d_model, vocab_size)
  
    def forward(self, x, tgt_mask):
        # Complete the forward pass
        x = self.embedding(x)
        x = self.positional_encoding(x)
        for layer in self.layers:
            x = layer(x,tgt_mask)
        x = self.fc(x)
        return F.log_softmax(x, dim=-1)

In [ ]:
# # Instantiate a decoder transformer and apply it to input_tokens and tgt_mask
# transformer_decoder = TransformerDecoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length)   
# output = transformer_decoder(input_tokens, tgt_mask)
# print(output)
# print(output.shape)